# Ontario Sunshine List — Gender Bias Analysis

**Pipeline:**
1. Load & normalize all CSV data (2017–2025)
2. Infer gender from first names (SSA lookup + char-gram model + optional HuggingFace model)
3. Cluster job titles via NLP (sentence embeddings + K-Means / HDBSCAN)
4. Statistical testing (Mann-Whitney U + Benjamini-Hochberg FDR)
5. Longitudinal trend analysis (salary growth, promotions, trajectory shapes)

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", "{:,.2f}".format)

## 1. Load Data

In [ ]:
from src.data_loader import load_all

df = load_all()
print(f"Total records: {len(df):,}")
print(f"Years:         {sorted(df['year'].dropna().unique().tolist())}")
print(f"Sectors:       {df['sector'].nunique()} unique")
print(f"Employers:     {df['employer'].nunique()} unique")
df.head()

In [ ]:
# Basic salary distribution overview
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

df.groupby("year")["salary"].median().plot(ax=axes[0], marker="o")
axes[0].set_title("Median Salary by Year")
axes[0].set_ylabel("Salary ($)")

df.groupby("year").size().plot(ax=axes[1], marker="o", color="steelblue")
axes[1].set_title("Number of Records by Year")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.show()

## 2. Gender Inference

**Setup (run once before this section):**
```bash
python scripts/download_gender_data.py   # downloads SSA names + NLTK corpus
python scripts/train_gender_model.py     # trains & caches char-gram model
```

To use a HuggingFace model as the third source, pass `hf_model_name=` below.  
Search [huggingface.co](https://huggingface.co/models?pipeline_tag=text-classification&q=gender+name) for a name→gender model and paste the model ID.

In [ ]:
from src.gender_inference import GenderClassifier

# Set hf_model_name to a HuggingFace model ID to enable the third source, e.g.:
# clf = GenderClassifier(hf_model_name="your-model-id-here")
clf = GenderClassifier()

gender_df = clf.classify_series(df["first_name_clean"])
df = df.join(gender_df)

print(df["gender"].value_counts())
print(f"\nCoverage (non-Uncertain): {(df['gender'] != 'Uncertain').mean():.1%}")

In [ ]:
# Gender distribution by year — records with confirmed gender only
confirmed = df[df["gender"] != "Uncertain"]

gender_by_year = (
    confirmed.groupby(["year", "gender"])
    .size()
    .unstack(fill_value=0)
    .assign(pct_female=lambda x: x["Female"] / (x["Female"] + x["Male"]))
)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

gender_by_year[["Female", "Male"]].plot(kind="bar", ax=axes[0], colormap="Set2")
axes[0].set_title("Record Count by Gender and Year")
axes[0].set_xlabel("")

gender_by_year["pct_female"].plot(ax=axes[1], marker="o", color="orchid")
axes[1].set_title("% Female on Sunshine List by Year")
axes[1].set_ylabel("Proportion Female")
axes[1].set_ylim(0, 1)
axes[1].axhline(0.5, color="grey", linestyle="--", linewidth=0.8)

plt.tight_layout()
plt.show()

## 3. Job Title Clustering (NLP)

*Coming next — sentence embeddings + K-Means / HDBSCAN with bootstrap stability testing.*

## 4. Statistical Testing

*Coming next — Mann-Whitney U per cluster, Benjamini-Hochberg FDR correction, raw gap + regression-adjusted gap.*

## 5. Longitudinal Trend Analysis

*Coming next — salary growth rates, cluster transitions (proxy for promotions), trajectory shapes, left-censoring discussion.*